In [ ]:
# ==========================================
# CELL 1: IMPORT THƯ VIỆN & CỐ ĐỊNH SEED
# ==========================================
import os
import time
import math
import random
import psutil
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from thop import profile # pip install thop
from tqdm.auto import tqdm
import gc
import warnings
warnings.filterwarnings('ignore')

# Module Routing (Cần có sẵn các file .py trong cùng thư mục)
from routing_smoe import SMoELayer
from routing_micro import MICROMoELayer
from routing_expert_choice import ExpertChoiceMoELayer
from routing_adaptive import AdaptiveDynamicMoELayer
from routing_deepseek import DeepSeekMoELayer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Đang chạy trên thiết bị: {device}")

def set_seed(seed):
    """Cố định seed để đảm bảo Reproducibility cho bài báo"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f" Đã thiết lập Seed = {seed}")

✅ Đang chạy trên thiết bị: cuda


In [ ]:
# ==========================================
# CELL 2: CẤU HÌNH HỆ THỐNG
# ==========================================
class Config:
    MODEL_NAME = "vinai/phobert-large"
    TRAIN_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\train.csv"
    VAL_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\validation.csv"
    TEST_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\test.csv"
    MAX_LEN = 256
    NUM_LABELS = 3
    
    BATCH_SIZE = 16
    LR = 2e-5
    EPOCHS = 20         # Tăng epoch tối đa lên để mô hình có không gian hội tụ
    PATIENCE = 3        # DỪNG SỚM: Số epoch tối đa chịu đựng nếu F1 không tăng
    
    NUM_EXPERTS = 8
    # Thêm SEEDS cho Multi-seed evaluation
    SEEDS = [42, 123, 999] 
    ROUTING_TYPES_TO_TEST = ["smoe", "micro", "expert_choice", "adaptive", "deepseek"]
    
    ROUTING_THRESHOLDS = 0.5 
    CAPACITY_FACTOR = 1.2 
    
    CHECKPOINT_DIR = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\excheckpoints_phobert"
    RESULTS_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\results_log_phobert.csv"
    SUMMARY_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\benchmark_summary.csv"

os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# ==========================================
# CELL 3: DATASET & DATALOADER (TỐI ƯU HÓA)
# ==========================================
label_map = {'entailment': 0, 'neutral': 1, 'contradiction': 2}
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)

class AdversarialNLIDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.labels = torch.tensor([label_map.get(str(l).strip().lower(), 1) for l in df['label']], dtype=torch.long)
        
        # Tokenize toàn bộ dataset một lần duy nhất vào RAM
        print(f" Đang pre-tokenize {len(df)} mẫu dữ liệu... Vui lòng đợi.")
        self.encodings = tokenizer(
            df['premise'].tolist(), 
            df['hypothesis'].tolist(),
            add_special_tokens=True,
            max_length=max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        print(" Pre-tokenize hoàn tất!")
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        # Chỉ trả về tensor đã lưu sẵn
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

# Đọc dữ liệu
df_train = pd.read_csv(Config.TRAIN_CSV).dropna().reset_index(drop=True)
df_val = pd.read_csv(Config.VAL_CSV).dropna().reset_index(drop=True)
df_test = pd.read_csv(Config.TEST_CSV).dropna().reset_index(drop=True)

# Khởi tạo DataLoader với pin_memory=True, BỎ num_workers
train_loader = DataLoader(
    AdversarialNLIDataset(df_train, tokenizer, Config.MAX_LEN), 
    batch_size=Config.BATCH_SIZE, 
    shuffle=True, 
    pin_memory=False
)
val_loader = DataLoader(
    AdversarialNLIDataset(df_val, tokenizer, Config.MAX_LEN), 
    batch_size=Config.BATCH_SIZE, 
    pin_memory=False
)
test_loader = DataLoader(
    AdversarialNLIDataset(df_test, tokenizer, Config.MAX_LEN), 
    batch_size=Config.BATCH_SIZE, 
    pin_memory=False
)

⏳ Đang pre-tokenize 8012 mẫu dữ liệu... Vui lòng đợi.
✅ Pre-tokenize hoàn tất!
⏳ Đang pre-tokenize 1000 mẫu dữ liệu... Vui lòng đợi.
✅ Pre-tokenize hoàn tất!
⏳ Đang pre-tokenize 1000 mẫu dữ liệu... Vui lòng đợi.
✅ Pre-tokenize hoàn tất!


In [ ]:
# ==========================================
# CELL 4: ARCHITECTURE & UTILITIES
# ==========================================
class CheckpointManager:
    def __init__(self, model, optimizer, scheduler, scaler, model_name="moe"):
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.scaler = scaler
        self.model_name = model_name
        self.best_checkpoint_path = os.path.join(Config.CHECKPOINT_DIR, f"{model_name}_best.pth")
        self.last_checkpoint_path = os.path.join(Config.CHECKPOINT_DIR, f"{model_name}_last.pth")
        self.results_path = Config.RESULTS_CSV
        
        if not os.path.exists(self.results_path):
            df = pd.DataFrame(columns=["Seed", "Epoch", "Routing", "Val_Acc", "Val_F1", "GFlops", "Runtime_ms", "VRAM_MB", "Entropy", "Expert_Usage"])
            df.to_csv(self.results_path, index=False)

    def save_checkpoint(self, epoch, val_f1, is_best=False):
        # CHỈ LƯU MODEL STATE để tiết kiệm 70% dung lượng ổ cứng
        state = {
            'epoch': epoch, 
            'model_state': self.model.state_dict(),
            'best_val_f1': val_f1
        }
        
        # Có thể thêm try-except để tránh sập toàn bộ pipeline nếu lỡ đầy ổ
        try:
            torch.save(state, self.last_checkpoint_path)
            if is_best: 
                torch.save(state, self.best_checkpoint_path)
        except Exception as e:
            print(f"❌ Lỗi khi lưu checkpoint: {e}")

    def load_checkpoint(self):
        start_epoch, best_val_f1 = 0, 0.0
        if os.path.exists(self.last_checkpoint_path):
            state = torch.load(self.last_checkpoint_path)
            
            # --- FIX LỖI THOP PROFILE ---
            # 1. Lọc bỏ toàn bộ các key do thop tự động sinh ra
            model_state = state['model_state']
            clean_state_dict = {k: v for k, v in model_state.items() if 'total_ops' not in k and 'total_params' not in k}
            
            # 2. Thêm strict=False để PyTorch phớt lờ nếu vẫn còn key thừa
            self.model.load_state_dict(clean_state_dict, strict=False)
            # -----------------------------
            
            # (Nếu ở bước tối ưu dung lượng trước đó bạn đã xóa phần load optimizer/scheduler thì không cần dòng này)
            if 'optimizer_state' in state:
                self.optimizer.load_state_dict(state['optimizer_state'])
                self.scheduler.load_state_dict(state['scheduler_state'])
                self.scaler.load_state_dict(state['scaler_state'])
                
            start_epoch = state['epoch'] + 1
            best_val_f1 = state.get('best_val_f1', 0.0)
            print(f" Đã khôi phục thành công! Bắt đầu từ Epoch {start_epoch + 1}")
        return start_epoch, best_val_f1

    def log_results(self, row):
        """Hàm lưu log được viết lại bằng pd.concat chuẩn"""
        df = pd.read_csv(self.results_path)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        df.to_csv(self.results_path, index=False)

def calculate_routing_metrics(model):
    """Tính toán Entropy và xuất mảng phân phối tần suất của Expert"""
    metrics = {"entropy": 0.0, "expert_usage_distribution": None}
    try:
        moe = model.moe_layer
        if hasattr(moe, "gate_logits") and moe.gate_logits is not None:
            probs = torch.softmax(moe.gate_logits, dim=-1)
            entropy = -(probs * torch.log(probs + 1e-9)).sum(dim=-1).mean()
            metrics["entropy"] = entropy.item()
            metrics["expert_usage_distribution"] = probs.mean(dim=0).cpu().numpy().tolist()
        elif hasattr(moe, "expert_usage") and moe.expert_usage is not None:
            usage = moe.expert_usage.float()
            probs = usage / (usage.sum() + 1e-9)
            entropy = -(probs * torch.log(probs + 1e-9)).sum()
            metrics["entropy"] = entropy.item()
            metrics["expert_usage_distribution"] = usage.cpu().numpy().tolist()
    except Exception:
        pass
    return metrics

class LayerAttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size), 
            nn.Tanh(), 
            nn.Linear(hidden_size, 1)
        )
        
    def forward(self, hidden_states, attention_mask):
        attn_weights = self.attention(hidden_states).squeeze(-1)
        min_val = torch.finfo(attn_weights.dtype).min 
        attn_weights = attn_weights.masked_fill(attention_mask == 0, min_val)
        attn_weights = F.softmax(attn_weights, dim=-1)
        return torch.bmm(attn_weights.unsqueeze(1), hidden_states).squeeze(1)

class UnifiedMoENLI(nn.Module):
    def __init__(self, config, routing_type="expert_choice"):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(config.MODEL_NAME)
        hidden_size = self.backbone.config.hidden_size
        
        # Đóng băng 12 layer đầu
        for name, param in self.backbone.named_parameters():
            if 'encoder.layer' in name and int(name.split('.')[2]) < 12:
                param.requires_grad = False

        # Khởi tạo linh hoạt, loại bỏ hardcode fallback
        if routing_type == "smoe": self.moe_layer = SMoELayer(hidden_size, config.NUM_EXPERTS)
        elif routing_type == "micro": self.moe_layer = MICROMoELayer(hidden_size)
        elif routing_type == "expert_choice": self.moe_layer = ExpertChoiceMoELayer(hidden_size, config.NUM_EXPERTS, config.CAPACITY_FACTOR)
        elif routing_type == "adaptive": self.moe_layer = AdaptiveDynamicMoELayer(hidden_size, config.NUM_EXPERTS, getattr(config, 'ROUTING_THRESHOLDS', 0.5))
        elif routing_type == "deepseek": self.moe_layer = DeepSeekMoELayer(hidden_size, num_shared_experts=2, num_routed_experts=config.NUM_EXPERTS-2)
        else: raise ValueError(f" Routing type '{routing_type}' không hợp lệ!")
            
        self.attention_pooling = LayerAttentionPooling(hidden_size)
        self.classifier = nn.Sequential(
            nn.Dropout(0.2), 
            nn.Linear(hidden_size, hidden_size // 2), 
            nn.GELU(), 
            nn.Linear(hidden_size // 2, config.NUM_LABELS)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        moe_output = self.moe_layer(outputs.last_hidden_state)
        
        # Xử lý trường hợp MoE layer trả về tuple (output, aux_loss)
        aux_loss = 0.0
        if isinstance(moe_output, tuple):
            moe_output, aux_loss = moe_output
            
        pooled_output = self.attention_pooling(moe_output, attention_mask)
        logits = self.classifier(pooled_output)
        
        return logits, aux_loss

In [ ]:
# ==========================================
# CELL 5: MEGA PIPELINE (TỐI ƯU HÓA TỐC ĐỘ)
# ==========================================
all_results = []

for seed in Config.SEEDS:
    set_seed(seed)
    
    for current_routing in Config.ROUTING_TYPES_TO_TEST:
        print(f"\n{'='*60}")
        print(f" SEED {seed} | ĐANG HUẤN LUYỆN ROUTING: {current_routing.upper()}")
        print(f"{'='*60}")
        
        model = UnifiedMoENLI(Config(), routing_type=current_routing).to(device)
        optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=Config.LR, weight_decay=0.01)
        total_steps = len(train_loader) * Config.EPOCHS
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
        scaler = GradScaler()
        criterion = nn.CrossEntropyLoss()
        
        model_name = f"phobert_{current_routing}_seed{seed}"
        checkpoint_manager = CheckpointManager(model, optimizer, scheduler, scaler, model_name=model_name)
        start_epoch, best_val_f1 = checkpoint_manager.load_checkpoint()

        # --- ĐO GFLOPS 1 LẦN DUY NHẤT ĐẦU MỖI ROUTING ---
        dummy_ids = torch.ones(1, Config.MAX_LEN, dtype=torch.long).to(device)
        dummy_mask = torch.ones(1, Config.MAX_LEN, dtype=torch.long).to(device)
        macs, _ = profile(model, inputs=(dummy_ids, dummy_mask), verbose=False)
        gflops = (macs * 2) / 1e9
        print(f" Tài nguyên ước tính: {gflops:.2f} GFlops")
        # --------------------------------------------------

        final_gflops = gflops
        final_runtime = 0
        final_vram = 0
        final_entropy = 0
        early_stop_counter = 0

        # --- TRAIN & VALIDATION LOOP ---
        # --- TRAIN LOOP CHUẨN HÓA ---
        for epoch in range(start_epoch, Config.EPOCHS):
            model.train()
            # Khởi tạo tqdm đúng vị trí
            train_iterator = tqdm(train_loader, desc=f"Ep {epoch+1}/{Config.EPOCHS} [Train]", leave=False)
            
            # Reset gradient trước khi vào epoch
            optimizer.zero_grad(set_to_none=True) 
            
            for batch in train_iterator:
                ids = batch['input_ids'].to(device, non_blocking=True)
                mask = batch['attention_mask'].to(device, non_blocking=True)
                labels = batch['labels'].to(device, non_blocking=True)
                
                with autocast():
                    logits, aux_loss = model(ids, mask)
                    
                    # Tính tổng loss bao gồm aux_loss (nếu có) để cân bằng expert
                    main_loss = criterion(logits, labels)
                    loss = main_loss + aux_loss # Có thể nhân hệ số alpha cho aux_loss nếu cần
                    
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                
                # Reset gradient NGAY SAU KHI update xong
                optimizer.zero_grad(set_to_none=True)
                
                # Cập nhật postfix đúng chỗ (bên trong vòng lặp)
                train_iterator.set_postfix(loss=f"{loss.item():.4f}")

           
            
            model.eval()
            val_preds, val_labels = [], []
            
            torch.cuda.synchronize()
            start_time = time.time()
            
            # TỐI ƯU SUY LUẬN BẰNG INFERENCE_MODE
            with torch.inference_mode(): 
                val_iterator = tqdm(val_loader, desc=f"[{current_routing.upper()}] Ep {epoch+1}/{Config.EPOCHS} [Val]", leave=False)
                for batch in val_iterator:
                    b_ids = batch['input_ids'].to(device, non_blocking=True)
                    b_mask = batch['attention_mask'].to(device, non_blocking=True)
                    b_labels = batch['labels'].to(device, non_blocking=True)
                    
                    with autocast():
                        logits, _ = model(b_ids, b_mask)
                    
                    val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                    val_labels.extend(b_labels.cpu().numpy())
            
            torch.cuda.synchronize()
            runtime_ms = ((time.time() - start_time) / len(val_loader)) * 1000
            
            vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
            val_acc = accuracy_score(val_labels, val_preds)
            val_f1 = f1_score(val_labels, val_preds, average='macro')
            routing_stats = calculate_routing_metrics(model)
            entropy_val = routing_stats["entropy"]
            expert_dist = routing_stats["expert_usage_distribution"]
            
            print(f"[{current_routing.upper()}] Ep {epoch+1} | Acc: {val_acc:.4f} | F1: {val_f1:.4f} | {gflops:.2f} GFlops | {runtime_ms:.2f} ms/b | VRAM: {vram_mb:.0f} MB")
            
            checkpoint_manager.log_results({
                "Seed": seed, "Epoch": epoch+1, "Routing": current_routing, 
                "Val_Acc": val_acc, "Val_F1": val_f1, "GFlops": gflops, 
                "Runtime_ms": runtime_ms, "VRAM_MB": vram_mb, "Entropy": entropy_val,
                "Expert_Usage": str(expert_dist)
            })
            
            is_best = val_f1 > best_val_f1
            if is_best: 
                best_val_f1 = val_f1
                final_runtime, final_vram, final_entropy = runtime_ms, vram_mb, entropy_val
                early_stop_counter = 0 
                print(" Validation F1 cải thiện, lưu Best Checkpoint.")
            else:
                early_stop_counter += 1
                print(f" Validation F1 không cải thiện. Early Stop Counter: {early_stop_counter}/{Config.PATIENCE}")
                
            checkpoint_manager.save_checkpoint(epoch, best_val_f1, is_best)
            
            if early_stop_counter >= Config.PATIENCE:
                print(f" Kích hoạt Early Stopping tại Epoch {epoch+1}!")
                break

        # --- ĐÁNH GIÁ TRÊN TẬP TEST ---
        print(f"\n Loading best checkpoint cho {current_routing.upper()} (Seed {seed})...")
        state = torch.load(checkpoint_manager.best_checkpoint_path, map_location=device)
        model.load_state_dict(state["model_state"])
        model.eval()
        
        test_preds, test_labels = [], []
        with torch.inference_mode(): # Dùng inference_mode thay vì no_grad
            for batch in test_loader:
                ids = batch["input_ids"].to(device, non_blocking=True)
                mask = batch["attention_mask"].to(device, non_blocking=True)
                labels = batch["labels"].to(device, non_blocking=True)
                
                with autocast():
                    logits, _ = model(ids, mask)
                
                pred = torch.argmax(logits, 1)
                test_preds.extend(pred.cpu().numpy())
                test_labels.extend(labels.cpu().numpy())

        test_acc = accuracy_score(test_labels, test_preds)
        test_f1 = f1_score(test_labels, test_preds, average="macro")
        print(f" TEST ACC = {test_acc:.4f} | TEST F1 = {test_f1:.4f}")
        
        all_results.append({
            "Seed": seed, "Routing": current_routing, "Test_ACC": test_acc, "Test_F1": test_f1,
            "GFLOPS": final_gflops, "Runtime": final_runtime, "VRAM": final_vram, "Entropy": final_entropy
        })
        
        del model, optimizer, scheduler, checkpoint_manager
        torch.cuda.empty_cache()
        gc.collect()

🌱 Đã thiết lập Seed = 42

🚀 SEED 42 | ĐANG HUẤN LUYỆN ROUTING: SMOE
🔄 Đã khôi phục thành công! Bắt đầu từ Epoch 2
📊 Tài nguyên ước tính: 159.56 GFlops


KeyboardInterrupt: 